In [ ]:
import re
import warnings
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, ttest_ind
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, ccf, grangercausalitytests
from statsmodels.regression.rolling import RollingOLS
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.tsa.statespace.structural import UnobservedComponents

warnings.filterwarnings("ignore")

In [ ]:
path = 'done_diplom_filled_v2_with_keyrate (1).xlsx'
df_raw = pd.read_excel(path)
print(df_raw.shape)
df_raw.head()

#просто чтобы потом сто раз не писать долгие
explicit_map = {'Дата': 'date',
    'Задолженность по предоставленным кредитам, млн руб., в том числе': 'total_debt',
    'просроченная задолженность по предоставленным кредитам, млн руб.': 'overdue_debt',
    'Средневзвешенная ставка по кредитам, выданным в течение месяца, %': 'avg_loan_rate',
    'Ключевая ставка, % годовых': 'cb_rate'}
used_names = set(explicit_map.values())

def sanitize_name(name: str) -> str:
    name = str(name).strip().lower()
    name = name.replace('%', 'pct')
    name = name.replace('№', 'num')
    name = re.sub(r'[^0-9a-zA-Zа-яА-Я_]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    if not name:
        name = 'col'
    return name

rename_map = {}
for col in df_raw.columns:
    if col in explicit_map:
        rename_map[col] = explicit_map[col]
    else:
        base = sanitize_name(col)
        candidate = base
        counter = 1
        while candidate in used_names:
            counter += 1
            candidate = f"{base}_{counter}"
        rename_map[col] = candidate
        used_names.add(candidate)

df = df_raw.rename(columns=rename_map).copy()

df['date'] = pd.to_datetime(df['date'])
# на всякий слукчай чтобды проблем не было
df = df.sort_values('date').reset_index(drop=True)

for col in df.columns:
    if col == 'date':
        continue
    df[col] = pd.to_numeric(
        df[col].astype(str)
              .str.replace(',', '.', regex=False)
              .str.replace('%', '', regex=False)
              .str.replace('\xa0', '', regex=False)
              .str.strip(),
        errors='coerce'
    )
# собственно сам таргет который я обозначил в работе
df['target_ratio'] = df['overdue_debt'] / df['total_debt']
df['rate_spread'] = df['avg_loan_rate'] - df['cb_rate']

# тоже обозночал в работе почему именно ffil бралд
numeric_cols = [c for c in df.columns if c != 'date']
df[numeric_cols] = df[numeric_cols].ffill()

df = df.dropna(subset=['target_ratio', 'rate_spread']).copy()

series = df.set_index('date')


(97, 48)


In [ ]:
selection_n_train = 85
selection_sample = series.iloc[:selection_n_train].copy()

excluded_from_selection = {
    'target_ratio',
    'rate_spread'
}

candidate_features = [
    col for col in selection_sample.columns
    if col not in excluded_from_selection and pd.api.types.is_numeric_dtype(selection_sample[col])
]

X_full = selection_sample[candidate_features].copy()
y_full = selection_sample['target_ratio'].copy()

X_full = X_full.replace([np.inf, -np.inf], np.nan)
y_full = y_full.replace([np.inf, -np.inf], np.nan)

valid_cols = []
for col in X_full.columns:
    s = X_full[col]
    if s.notna().sum() == 0:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    valid_cols.append(col)
X_full = X_full[valid_cols].copy()

valid_rows = X_full.notna().all(axis=1) & y_full.notna()
X_full = X_full.loc[valid_rows].copy()
y_full = y_full.loc[valid_rows].copy()


base_splitter = TimeSeriesSplit(n_splits=5)
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],cv=base_splitter,random_state=0,max_iter=50000))])
base_pipeline.fit(X_full, y_full)
base_model = base_pipeline.named_steps['enet']
start_fracs = [0.60, 0.70, 0.80, 0.90, 1.00]
selection_records = []

for frac in start_fracs:
    end_idx = max(30, int(len(X_full) * frac))
    X_sub = X_full.iloc[:end_idx].copy()
    y_sub = y_full.iloc[:end_idx].copy()
    if len(X_sub) < 25:
        continue

    n_splits_sub = min(5, max(2, len(X_sub) // 12))
    sub_splitter = TimeSeriesSplit(n_splits=n_splits_sub)
    sub_pipeline =Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],cv=base_splitter,random_state=0,max_iter=50000))])
    sub_pipeline.fit(X_sub, y_sub)
    sub_model = sub_pipeline.named_steps['enet']
    coef = pd.Series(sub_model.coef_, index=X_sub.columns)
    selection_records.append(pd.DataFrame({
        'feature': coef.index,'coef': coef.values,'abs_coef': coef.abs().values,'selected': (coef.abs() > 1e-10).astype(int),'window_frac': frac}))

stability_df = pd.concat(selection_records, ignore_index=True)
stability_summary = (
    stability_df.groupby('feature', as_index=False).agg(selection_frequency=('selected', 'mean'),mean_abs_coef=('abs_coef', 'mean'),max_abs_coef=('abs_coef', 'max')))

base_coef_df = pd.DataFrame({'feature': X_full.columns,'base_coef': base_model.coef_,'base_abs_coef': np.abs(base_model.coef_)})

feature_ranking = (stability_summary.merge(base_coef_df, on='feature', how='left').sort_values(['selection_frequency', 'mean_abs_coef', 'base_abs_coef'],ascending=[False, False, False]).reset_index(drop=True))

selected_core_features = feature_ranking.head(10)['feature'].tolist()
selected_features = ['rate_spread'] + selected_core_features



feature_ranking.head(15)

,feature,selection_frequency,mean_abs_coef,max_abs_coef,base_coef,base_abs_coef
0,total_debt,1.0,0.000876,0.001085,-0.000703,0.000703
1,participants_количество_действующих_кредитных_...,1.0,0.000671,0.000957,0.000957,0.000957
2,overdue_debt,1.0,0.000335,0.000524,0.000524,0.000524
3,earlyrep_отступными,1.0,0.000045,0.000059,0.000035,0.000035
4,зп_номин_руб,1.0,0.000043,0.000084,-0.000084,0.000084
5,безработные_15_72_тыс,1.0,0.000039,0.000048,0.000040,0.000040
6,количество_предоставленных_кредитов_за_месяц_е...,1.0,0.000005,0.000008,0.000008,0.000008
7,participants_количество_кредитных_организаций_...,0.8,0.000211,0.000336,0.000177,0.000177
8,цена_квм_низкого_качества_вторичный_рынок_жилья,0.8,0.000143,0.000305,-0.000305,0.000305
9,средневзвешенный_срок_кредитования_по_кредитам...,0.8,0.000121,0.000222,-0.000161,0.000161


In [ ]:
selected_features = ['rate_spread', 'total_debt', 'participants_количество_действующих_кредитных_организаций', 'overdue_debt', 'earlyrep_отступными', 'зп_номин_руб', 'безработные_15_72_тыс', 'количество_предоставленных_кредитов_за_месяц_единиц', 'participants_количество_кредитных_организаций_предоставляющих_жилищные_кредиты', 'цена_квм_низкого_качества_вторичный_рынок_жилья', 'средневзвешенный_срок_кредитования_по_кредитам_выданным_в_течение_месяца_месяцев']

FINAL_MODEL_CONFIG = {
    'ARIMAX': {'order': (3, 1, 1), 'trend': 'c'},'VAR': {'diff_order': 1,'lags': 3},
    'VECM': {'k_ar_diff': 3,'coint_rank': 1,'deterministic': 'co'},
    'ARDL': {'p_lag': 3,'q_lag': 0,'causal': False,'feat_cols': selected_features.copy()},'UCM': {'level': 'local linear trend','seasonal': 24},
    'TVP-ARX': {'lambda': 0.99,'y_lags': 1,'x_lag': 0,'ridge_scale': 25.0,'feat_cols': selected_features[:6].copy()}}
display(pd.DataFrame(FINAL_MODEL_CONFIG).T)


,order,trend,diff_order,lags,k_ar_diff,coint_rank,deterministic,p_lag,q_lag,causal,feat_cols,level,seasonal,lambda,y_lags,x_lag,ridge_scale
ARIMAX,"(3, 1, 1)",c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VAR,NaN,NaN,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VECM,NaN,NaN,NaN,NaN,3,1,co,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ARDL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0,False,"[rate_spread, total_debt, participants_количес...",NaN,NaN,NaN,NaN,NaN,NaN
UCM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,local linear trend,24,NaN,NaN,NaN,NaN
TVP-ARX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[rate_spread, total_debt, participants_количес...",NaN,NaN,0.99,1,0,25.0


In [ ]:

model_cols = ['target_ratio'] + selected_features
model_df = series[model_cols].copy()

model_df = model_df.replace([np.inf, -np.inf], np.nan)


model_df = model_df.ffill()


model_df = model_df.dropna(axis=0, how='any').copy()


display(model_df.head())


,target_ratio,rate_spread,total_debt,participants_количество_действующих_кредитных_организаций,overdue_debt,earlyrep_отступными,зп_номин_руб,безработные_15_72_тыс,количество_предоставленных_кредитов_за_месяц_единиц,participants_количество_кредитных_организаций_предоставляющих_жилищные_кредиты,цена_квм_низкого_качества_вторичный_рынок_жилья,средневзвешенный_срок_кредитования_по_кредитам_выданным_в_течение_месяца_месяцев
date,,,,,,,,,,,,
2018-04-01,0.011184,2.39,5408938,542.0,60491,124.0,43381.0,3789.950,119087,403.0,45430.05,193.1
2018-05-01,0.011049,2.39,5527172,542.0,61069,124.0,44076.0,3723.716,125729,403.0,45430.05,189.4
2018-06-01,0.010995,2.30,5628078,542.0,61882,124.0,45848.0,3633.506,118538,403.0,45430.05,192.0
2018-07-01,0.010765,2.24,5744447,524.0,61838,185.0,42413.0,3597.955,121007,395.0,46164.40,193.1
2018-08-01,0.010787,2.33,5852754,524.0,63135,185.0,41364.0,3565.295,122212,395.0,46164.40,200.0


In [ ]:

n_train = 85
train = model_df.iloc[:n_train].copy()
test  = model_df.iloc[n_train:].copy()
endog_train = train['target_ratio'].copy()
endog_test  = test['target_ratio'].copy()
exog_train = train[selected_features].copy()
exog_test  = test[selected_features].copy()
var_cols = ['target_ratio'] + selected_features
var_train = train[var_cols].copy()
var_test  = test[var_cols].copy()

In [ ]:
def safe_kpss(series):
    try:
        stat, pvalue, _, _ = kpss(series, regression='c', nlags='auto')
        return stat, pvalue
    except Exception:
        return np.nan, np.nan

def stationarity_test(series):
    s = pd.Series(series).dropna().astype(float)
    if len(s) < 12:
        return {'adf_pvalue': np.nan, 'kpss_pvalue': np.nan, 'stationary': False}
    try:
        adf_pvalue = adfuller(s, autolag='AIC')[1]
    except Exception:
        adf_pvalue = np.nan
    _, kpss_pvalue = safe_kpss(s)
    stationary = False
    if not np.isnan(adf_pvalue) and not np.isnan(kpss_pvalue):
        stationary = (adf_pvalue < 0.05) and (kpss_pvalue > 0.05)
    elif not np.isnan(adf_pvalue):
        stationary = adf_pvalue < 0.05
    return {'adf_pvalue': adf_pvalue, 'kpss_pvalue': kpss_pvalue, 'stationary': stationary}

def choose_integration_order(series, max_d=1):
    current = pd.Series(series).dropna().astype(float)
    diagnostics = []
    for d in range(max_d + 1):
        stat = stationarity_test(current)
        diagnostics.append({'diff_order': d,'adf_pvalue': stat['adf_pvalue'],'kpss_pvalue': stat['kpss_pvalue'],'stationary': stat['stationary']})
        if stat['stationary']:
            return d, pd.DataFrame(diagnostics)
        current = current.diff().dropna()
    return max_d, pd.DataFrame(diagnostics)

def make_stationarity_table(df, max_d=1):
    rows = []
    for col in df.columns:
        level_stat = stationarity_test(df[col])
        chosen_d, _ = choose_integration_order(df[col], max_d=max_d)
        rows.append({'series': col,'level_adf_pvalue': level_stat['adf_pvalue'],'level_kpss_pvalue': level_stat['kpss_pvalue'],'stationary_in_levels': level_stat['stationary'],'chosen_diff_order': chosen_d})
    return pd.DataFrame(rows).sort_values(['chosen_diff_order', 'series']).reset_index(drop=True)

def difference_series_train_test(train_series, test_series, diff_order=0):
    train_series = pd.Series(train_series).astype(float).copy()
    test_series = pd.Series(test_series).astype(float).copy()
    if diff_order <= 0:
        return train_series, test_series
    combined = pd.concat([train_series, test_series], axis=0)
    transformed = combined.copy()
    for _ in range(int(diff_order)):
        transformed = transformed.diff()
    train_out = transformed.loc[train_series.index].dropna().astype(float)
    test_out = transformed.loc[test_series.index].dropna().astype(float)
    return train_out, test_out

def build_stationary_train_test(train_df, test_df, diff_order_map):
    train_parts = []
    test_parts = []
    for col in train_df.columns:
        d = int(diff_order_map.get(col, 0))
        tr_col, te_col = difference_series_train_test(train_df[col], test_df[col], diff_order=d)
        train_parts.append(tr_col.rename(col))
        test_parts.append(te_col.rename(col))
    train_out = pd.concat(train_parts, axis=1).dropna()
    test_out = pd.concat(test_parts, axis=1).dropna()
    return train_out, test_out

def align_target_and_exog(target_series, exog_df):
    idx = pd.Index(target_series.index).intersection(exog_df.index)
    return target_series.loc[idx].astype(float), exog_df.loc[idx].astype(float)

def forecast_from_differences(last_level, diff_forecast):
    levels = []
    current = float(last_level)
    for value in np.asarray(diff_forecast, dtype=float).ravel():
        current += value
        levels.append(current)
    return np.array(levels, dtype=float)

def build_nan_forecast(index, fill_value=np.nan):
    return pd.Series(fill_value, index=index, dtype=float)

def safe_metric_frame(y_true, y_pred):
    return pd.concat([pd.Series(y_true, name='y_true').astype(float),pd.Series(y_pred, name='y_pred').astype(float)], axis=1).dropna()

stationarity_train_df = make_stationarity_table(train[var_cols], max_d=1)
integration_orders = stationarity_train_df.set_index('series')['chosen_diff_order'].astype(int).to_dict()

exog_diff_order_map = {col: int(integration_orders.get(col, 0)) for col in selected_features}
target_diff_order = int(integration_orders.get('target_ratio', 0))

endog_train_stationary, endog_test_stationary = difference_series_train_test(
    endog_train, endog_test, diff_order=target_diff_order)
exog_train_stationary, exog_test_stationary = build_stationary_train_test(
    exog_train, exog_test, diff_order_map=exog_diff_order_map)

endog_train_stationary, exog_train_stationary = align_target_and_exog(endog_train_stationary, exog_train_stationary)
endog_test_stationary, exog_test_stationary = align_target_and_exog(endog_test_stationary, exog_test_stationary)
arima_train_df = pd.concat([endog_train.rename('target_ratio'), exog_train_stationary], axis=1).dropna()
arima_endog_train = arima_train_df['target_ratio'].astype(float)
arima_exog_train = arima_train_df[selected_features].astype(float)
arima_exog_test = exog_test_stationary[selected_features].astype(float)
reg_train_df = pd.concat([endog_train_stationary.rename('target_ratio'), exog_train_stationary], axis=1).dropna()
reg_test_df = pd.concat([endog_test_stationary.rename('target_ratio'), exog_test_stationary], axis=1).dropna()
reg_endog_train = reg_train_df['target_ratio'].astype(float)
reg_exog_train = reg_train_df[selected_features].astype(float)
reg_endog_test = reg_test_df['target_ratio'].astype(float)
reg_exog_test = reg_test_df[selected_features].astype(float)
display(stationarity_train_df)


,series,level_adf_pvalue,level_kpss_pvalue,stationary_in_levels,chosen_diff_order
0,количество_предоставленных_кредитов_за_месяц_е...,0.045902,0.10,True,0
1,earlyrep_отступными,0.125752,0.01,False,1
2,overdue_debt,0.828501,0.10,False,1
3,participants_количество_действующих_кредитных_...,0.088313,0.01,False,1
4,participants_количество_кредитных_организаций_...,0.026080,0.01,False,1
5,rate_spread,0.911221,0.01,False,1
6,target_ratio,0.562387,0.01,False,1
7,total_debt,0.982723,0.01,False,1
8,безработные_15_72_тыс,0.922871,0.01,False,1
9,зп_номин_руб,0.865303,0.01,False,1


## ARIMAX

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
arimax_model = SARIMAX(arima_endog_train,exog=arima_exog_train,order=FINAL_MODEL_CONFIG['ARIMAX']['order'],seasonal_order=(0, 0, 0, 0),
    trend=FINAL_MODEL_CONFIG['ARIMAX']['trend'],enforce_stationarity=False,enforce_invertibility=False)
arimax_res = arimax_model.fit(disp=False, maxiter=200)
arimax_forecast = pd.Series(arimax_res.forecast(steps=len(arima_exog_test), exog=arima_exog_test),index=arima_exog_test.index).astype(float)




## VAR

In [ ]:
var_system_cols = ['target_ratio'] + selected_features
var_stationarity_df = stationarity_train_df.copy()
best_var_diff_order = FINAL_MODEL_CONFIG['VAR']['diff_order']
best_var_lag_order = FINAL_MODEL_CONFIG['VAR']['lags']

if best_var_diff_order == 0:
    var_model_data = var_train[var_system_cols].copy()
    var_model = VAR(var_model_data)
    var_res = var_model.fit(best_var_lag_order)
    var_pred = var_res.forecast(var_model_data.values[-var_res.k_ar:], steps=len(test))
    var_forecast = pd.Series(var_pred[:, 0], index=test.index).astype(float)
else:
    var_model_data = var_train[var_system_cols].diff(best_var_diff_order).dropna()
    var_model = VAR(var_model_data)
    var_res = var_model.fit(best_var_lag_order)
    var_pred_diff = var_res.forecast(var_model_data.values[-var_res.k_ar:], steps=len(test))
    var_forecast = pd.Series(forecast_from_differences(var_train['target_ratio'].iloc[-1], var_pred_diff[:, 0]),index=test.index).astype(float)

var_is_stable = bool(var_res.is_stable())



## VECM

In [ ]:

vecm_model = VECM(var_train[['target_ratio'] + selected_features], k_ar_diff=FINAL_MODEL_CONFIG['VECM']['k_ar_diff'],coint_rank=FINAL_MODEL_CONFIG['VECM']['coint_rank'],deterministic=FINAL_MODEL_CONFIG['VECM']['deterministic'])
vecm_res = vecm_model.fit()
vecm_pred = vecm_res.predict(steps=len(test))
vecm_forecast = pd.Series(vecm_pred[:, 0], index=test.index).astype(float)



## ARDL

In [ ]:

ardl_feat_cols = FINAL_MODEL_CONFIG['ARDL']['feat_cols']
ardl_model = ARDL(endog_train,lags=FINAL_MODEL_CONFIG['ARDL']['p_lag'],exog=exog_train[ardl_feat_cols],order={feature: FINAL_MODEL_CONFIG['ARDL']['q_lag'] for feature in ardl_feat_cols},trend='c',causal=FINAL_MODEL_CONFIG['ARDL']['causal'])
ardl_res = ardl_model.fit()
ardl_forecast = pd.Series(ardl_res.forecast(steps=len(test), exog=exog_test[ardl_feat_cols]),index=test.index).astype(float)






## UCM

In [ ]:

ucm_model = UnobservedComponents(endog_train.astype(float),level=FINAL_MODEL_CONFIG['UCM']['level'],seasonal=FINAL_MODEL_CONFIG['UCM']['seasonal'],exog=exog_train.astype(float))
ucm_res = ucm_model.fit(disp=False, maxiter=500)
ucm_forecast = pd.Series(ucm_res.forecast(steps=len(test), exog=exog_test.astype(float)),index=test.index,name='UCM').astype(float)



## TVP-ARX

In [ ]:


def fit_tvp_arx_rls(y_train, X_train, lam=0.99, p_y=1, x_lag=0, ridge_scale=25.0):
    y_train = pd.Series(y_train).astype(float).copy()
    X_train = pd.DataFrame(X_train).astype(float).copy()
    feature_cols = list(X_train.columns)

    max_lag = max(p_y, x_lag)


    y_mean = float(y_train.mean())
    y_std = float(y_train.std(ddof=0))
    if y_std == 0 or np.isnan(y_std):
        y_std = 1.0

    x_mean = X_train.mean()
    x_std = X_train.std(ddof=0).replace(0, 1.0).fillna(1.0)

    y_scaled = (y_train - y_mean) / y_std
    X_scaled = (X_train - x_mean) / x_std

    design_rows = []
    targets = []
    for t in range(max_lag, len(y_train)):
        row = [1.0]
        for lag in range(1, p_y + 1):
            row.append(float(y_scaled.iloc[t - lag]))
        for feature in feature_cols:
            x_pos = t - x_lag
            row.append(float(X_scaled[feature].iloc[x_pos]))
        design_rows.append(row)
        targets.append(float(y_scaled.iloc[t]))

    Z = np.asarray(design_rows, dtype=float)
    target_arr = np.asarray(targets, dtype=float)

    k = Z.shape[1]
    beta_path = np.zeros((len(target_arr), k))
    P = np.eye(k) * ridge_scale

    for t in range(len(target_arr)):
        x_t = Z[t]
        y_t = target_arr[t]
        beta_prev = np.zeros(k) if t == 0 else beta_path[t - 1].copy()
        R = P @ x_t
        S = lam + x_t.T @ R
        K = R / S
        error_t = y_t - x_t @ beta_prev
        beta_curr = beta_prev + K * error_t
        P = (P - np.outer(K, R)) / lam
        beta_path[t] = beta_curr

    return {'beta_path': beta_path,'beta_final': beta_path[-1].copy(),'feature_cols': feature_cols,'p_y': p_y,'x_lag': x_lag,'lam': lam,'ridge_scale': ridge_scale,
            'y_mean': y_mean,'y_std': y_std,'x_mean': x_mean,'x_std': x_std}

def tvp_arx_forecast(state, y_history, X_history, X_future):
    y_history = pd.Series(y_history).astype(float).copy()
    X_history = pd.DataFrame(X_history).astype(float).copy()
    X_future = pd.DataFrame(X_future).astype(float).copy()

    feature_cols = state['feature_cols']
    beta_final = state['beta_final']
    p_y = state['p_y']
    x_lag = state['x_lag']
    y_mean = state['y_mean']
    y_std = state['y_std']
    x_mean = state['x_mean']
    x_std = state['x_std']

    exog_combined = pd.concat([X_history[feature_cols], X_future[feature_cols]], axis=0)
    y_buffer = list(y_history.values.astype(float))
    preds = []

    for h in range(len(X_future)):
        row = [1.0]
        for lag in range(1, p_y + 1):
            y_lag_value = y_buffer[-lag]
            row.append((y_lag_value - y_mean) / y_std)
        for feature in feature_cols:
            combined_pos = len(X_history) + h - x_lag
            x_value = float(exog_combined[feature].iloc[combined_pos])
            row.append((x_value - x_mean[feature]) / x_std[feature])
        row = np.asarray(row, dtype=float)
        pred_scaled = float(row @ beta_final)
        pred_level = pred_scaled * y_std + y_mean
        preds.append(pred_level)
        y_buffer.append(pred_level)

    return pd.Series(preds, index=X_future.index)

tvp_feat_cols = FINAL_MODEL_CONFIG['TVP-ARX']['feat_cols']
tvp_state = fit_tvp_arx_rls( reg_endog_train,reg_exog_train[tvp_feat_cols],lam=FINAL_MODEL_CONFIG['TVP-ARX']['lambda'],p_y=FINAL_MODEL_CONFIG['TVP-ARX']['y_lags'],
    x_lag=FINAL_MODEL_CONFIG['TVP-ARX']['x_lag'],ridge_scale=FINAL_MODEL_CONFIG['TVP-ARX']['ridge_scale'])

tvp_forecast_stationary = tvp_arx_forecast(state=tvp_state,y_history=reg_endog_train,X_history=reg_exog_train[tvp_feat_cols],
    X_future=reg_exog_test[tvp_feat_cols]).astype(float)

if target_diff_order == 0:
    tvp_forecast = tvp_forecast_stationary.reindex(test.index)
else:
    tvp_forecast = pd.Series(forecast_from_differences(endog_train.iloc[-1], tvp_forecast_stationary.values),index=tvp_forecast_stationary.index).reindex(test.index)



In [ ]:

model_preds = {'ARIMAX': arimax_forecast,'VAR': var_forecast,'VECM': vecm_forecast,'ARDL': ardl_forecast,
  'UCM': ucm_forecast,'TVP-ARX': tvp_forecast}

metrics = []
for name, pred in model_preds.items():
    aligned = safe_metric_frame(endog_test, pred)
    if aligned.empty:
        metrics.append({'Model': name, 'MAE': np.nan, 'RMSE': np.nan, 'n_eval': 0})
        continue
    mae = mean_absolute_error(aligned['y_true'], aligned['y_pred'])
    rmse_value = np.sqrt(mean_squared_error(aligned['y_true'], aligned['y_pred']))
    metrics.append({'Model': name, 'MAE': mae, 'RMSE': rmse_value, 'n_eval': len(aligned)})

metrics_df = pd.DataFrame(metrics).sort_values('RMSE', na_position='last').reset_index(drop=True)
display(metrics_df)


,Model,MAE,RMSE,n_eval
0,UCM,0.000061,0.000072,9
1,VECM,0.000249,0.000277,9
2,ARIMAX,0.000261,0.000289,9
3,VAR,0.000295,0.000313,9
4,ARDL,0.000481,0.000619,9
5,TVP-ARX,0.000681,0.000792,9


In [ ]:


plot_metrics = metrics_df.set_index('Model')[['MAE', 'RMSE']]

fig, axes = plt.subplots(4, 2, figsize=(15, 18), sharex=True)
axes = axes.flatten()

for ax, (name, pred) in zip(axes, model_preds.items()):
    ax.plot(train.index, endog_train, label='тренировочные реальные', color='blue')
    ax.plot(test.index, endog_test, label='тренировочные реальные', color='black', linewidth=2)
    ax.plot(pd.Series(pred).index, pd.Series(pred).values, label=f'{name} forecast', color='red')
    ax.axvline(test.index[0], color='gray', linestyle='--', linewidth=1)
    ax.set_title(name)

    if name in plot_metrics.index:
        mae_val = plot_metrics.loc[name, 'MAE']
        rmse_val = plot_metrics.loc[name, 'RMSE']
        label_text = f"MAE = {mae_val:.6f}\nRMSE = {rmse_val:.6f}"
        ax.text(
            0.02, 0.98,
            label_text,
            transform=ax.transAxes,
            ha='left', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, linestyle='--')

for ax in axes[len(model_preds):]:
    ax.axis('off')

fig.suptitle('сравнение прогнозов', fontsize=16, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.985])

figure_path = 'final_model_сравнение.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()


display(metrics_df)


,Model,MAE,RMSE,n_eval
0,UCM,0.000061,0.000072,9
1,VECM,0.000249,0.000277,9
2,ARIMAX,0.000261,0.000289,9
3,VAR,0.000295,0.000313,9
4,ARDL,0.000481,0.000619,9
5,TVP-ARX,0.000681,0.000792,9


In [ ]:

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-12, denom)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

def mape_safe(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return 100 * np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps)))

def directional_accuracy(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)

    dy_true = y_true.diff().dropna()
    dy_pred = y_pred.diff().dropna()

    common_idx = dy_true.index.intersection(dy_pred.index)
    if len(common_idx) == 0:
        return np.nan

    s_true = np.sign(dy_true.loc[common_idx])
    s_pred = np.sign(dy_pred.loc[common_idx])
    return 100 * np.mean(s_true == s_pred)





def to_aligned_series(pred, target_index, model_name):


    if isinstance(pred, pd.DataFrame):
        pred = pred.iloc[:, 0]

    if isinstance(pred, pd.Series):
        pred_values = pred.astype(float).to_numpy().reshape(-1)
    else:
        pred_values = np.asarray(pred, dtype=float).reshape(-1)

    min_len = min(len(pred_values), len(target_index))


    return pd.Series(pred_values[:min_len], index=target_index[:min_len], name=model_name).astype(float)

def safe_metric_frame(y_true, y_pred):
    tmp = pd.concat([pd.Series(y_true, name='y_true').astype(float),pd.Series(y_pred, name='y_pred').astype(float)], axis=1).dropna()
    return tmp
candidate_models = [("ARIMAX", ["arimax_forecast", "arima_forecast"]),("VAR", ["var_forecast"]),("VECM", ["vecm_forecast"]),("ARDL", ["ardl_forecast"]),("UCM", ["ucm_forecast"]),
 ("TVP-ARX", ["tvp_forecast"]),]

model_preds = {}
missing_models = []

for pretty_name, var_candidates in candidate_models:
    found = False
    for var_name in var_candidates:
        if var_name in globals():
                s = to_aligned_series(globals()[var_name], test.index, pretty_name)
                model_preds[pretty_name] = s
                found = True
                break
    if not found:
        missing_models.append(pretty_name)

metrics_rows = []
aligned_store = {}

for model_name, pred in model_preds.items():
    aligned = safe_metric_frame(endog_test, pred)

    if aligned.empty:
        metrics_rows.append({"Model": model_name,"N_used": 0,"MAE": np.nan,"RMSE": np.nan,"sMAPE_%": np.nan,"MAPE_safe_%": np.nan,"Directional_Accuracy_%": np.nan})
        continue

    aligned_store[model_name] = aligned.copy()

    metrics_rows.append({"Model": model_name,"N_used": len(aligned),
        "MAE": mean_absolute_error(aligned["y_true"], aligned["y_pred"]),"RMSE": rmse(aligned["y_true"], aligned["y_pred"]),"sMAPE_%": smape(aligned["y_true"], aligned["y_pred"]),
        "MAPE_safe_%": mape_safe(aligned["y_true"], aligned["y_pred"]),"Directional_Accuracy_%": directional_accuracy(aligned["y_true"], aligned["y_pred"])})

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE", na_position="last").reset_index(drop=True)
metrics_df["ранжировка_RMSE"] = np.arange(1, len(metrics_df) + 1)
metrics_df = metrics_df[
    ["ранжировка_RMSE", "Model", "N_used", "MAE", "RMSE", "sMAPE_%", "MAPE_safe_%", "Directional_Accuracy_%"]]

display(metrics_df)

forecast_table = pd.DataFrame(index=test.index)
forecast_table["Actual"] = endog_test.astype(float)

for model_name, pred in model_preds.items():
    aligned_pred = to_aligned_series(pred, test.index, model_name)
    forecast_table[model_name] = aligned_pred.reindex(test.index)

display(forecast_table.head())
error_table = pd.DataFrame(index=test.index)
error_table["Actual"] = endog_test.astype(float)

for model_name in model_preds.keys():
    pred_aligned_full = forecast_table[model_name]
    error_table[f"{model_name}Err"] = error_table["Actual"] - pred_aligned_full
    error_table[f"{model_name}AE"] = np.abs(error_table["Actual"] - pred_aligned_full)

display(error_table.head())
sample_info = pd.DataFrame({"Block": ["Full sample", "Train", "Test"],"Start": [series.index.min(), train.index.min(), test.index.min()],"End": [series.index.max(), train.index.max(), test.index.max()],
                            "Observations": [len(series), len(train), len(test)]})

desc_vars = ["target_ratio", "rate_spread"]
desc_vars = [v for v in desc_vars if v in series.columns]

desc_full = series[desc_vars].describe().T
desc_train = train[desc_vars].describe().T
desc_test = test[desc_vars].describe().T

display(sample_info)
display(desc_full)


'''
Код ниже был написан нейросетью
Также названия были скорректированы мною , то есть код сгенерированный с моими правками
Код был сгенерирован по промту : исходя из кода что я тебе прислал, сделай мне красивые графики и сохрани их для того , чтобы я мог использовать их  в своей ВКР, а также сохрани итоговые резуьлтаты моделирования в отдельные удобные файлы +2 запрос, сделай мне еще график , этих пока недостаточно

Модель: deepseek,expert,deepthink https://chat.deepseek.com

Работа выполнена правильно на 7 из 10, не все графики поулчились очень красивыми и удобными для вставки в вкр
'''

import os
from pathlib import Path
OUTPUT_DIR = Path("article_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def make_adf_table(series_dict):
    rows = []
    for name, s in series_dict.items():
        s = pd.Series(s).dropna().astype(float)
        adf_res = adfuller(s, autolag="AIC")
        try:
            kpss_res = kpss(s, regression="c", nlags="auto")
            kpss_stat, kpss_p = kpss_res[0], kpss_res[1]
        except Exception:
            kpss_stat, kpss_p = np.nan, np.nan

        rows.append({"Series": name,"ADF statistic": adf_res[0],"ADF p-value": adf_res[1],"KPSS statistic": kpss_stat,"KPSS p-value": kpss_p,"N obs": len(s)})
    return pd.DataFrame(rows)
def save_latex_table(df, path, caption=None, label=None, float_format="%.6f"):
    fmt = float_format.__mod__ if isinstance(float_format, str) else None
    latex = df.to_latex(index=True, escape=False, float_format=fmt)

    if caption or label:
        wrapper = []
        wrapper.append("\\begin{table}[!htbp]\n\\centering\n")
        if caption:
            wrapper.append(f"\\caption{{{caption}}}\n")
        if label:
            wrapper.append(f"\\label{{{label}}}\n")
        wrapper.append(latex)
        wrapper.append("\n\\end{table}\n")
        latex = "".join(wrapper)

    with open(path, "w", encoding="utf-8") as f:
        f.write(latex)
stationarity_dict = {"target_ratio_level": series["target_ratio"],"target_ratio_diff1": series["target_ratio"].diff(),
"target_ratio_diff2": series["target_ratio"].diff().diff(),}

if "rate_spread" in series.columns:
    stationarity_dict["rate_spread_level"] = series["rate_spread"]

stationarity_df = make_adf_table(stationarity_dict)
display(stationarity_df)
overlay_path = OUTPUT_DIR / "figure_1.png"

plt.figure(figsize=(13, 7))
plt.plot(train.index, endog_train, label="Тренировочный")
plt.plot(test.index, endog_test, label="Тест ", linewidth=2)
forecast_plot = forecast_table.drop(columns=["Actual", "ARDL-ECM"], errors="ignore")

for model_name, pred in forecast_plot.items():
    plt.plot(pred.index, pred.values, label=model_name)

plt.axvline(test.index[0], linestyle="--")
plt.title("Сравнение прогнозов по моделям")
plt.xlabel("Дата")
plt.ylabel("Таргет")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(overlay_path, dpi=300, bbox_inches="tight")
plt.show()
subplots_path = OUTPUT_DIR / "figure_2.png"

n_models = len(model_preds)
ncols = 2
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
axes = np.array(axes).reshape(-1)

for ax, model_name in zip(axes, model_preds.keys()):
    pred = forecast_table[model_name]
    ax.plot(train.index, endog_train, label="Тренировочный")
    ax.plot(test.index, endog_test, label="Тест", linewidth=2)
    ax.plot(pred.index, pred.values, label=model_name)
    ax.axvline(test.index[0], linestyle="--")
    ax.set_title(model_name)
    ax.legend(fontsize=7)

for j in range(len(model_preds), len(axes)):
    axes[j].axis("off")

fig.tight_layout()
fig.savefig(subplots_path, dpi=300, bbox_inches="tight")
plt.show()
metrics_bar_path = OUTPUT_DIR / "figure_3.png"

plot_df = metrics_df.sort_values("RMSE", na_position="last")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_df["Model"], plot_df["RMSE"])
axes[0].set_title("RMSE в разбивке на модели")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(plot_df["Model"], plot_df["MAE"])
axes[1].set_title("MAE в разбивке на модели")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(metrics_bar_path, dpi=300, bbox_inches="tight")
plt.show()
abs_error_path = OUTPUT_DIR / "figure_4.png"

plt.figure(figsize=(13, 7))
for model_name in model_preds.keys():
    plt.plot(error_table.index, error_table[f"{model_name}AE"], label=model_name)

plt.title("Абсолютное предсказание на тестовом вериоде")
plt.xlabel("Дата")
plt.ylabel("Абсолютная ошибка")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(abs_error_path, dpi=300, bbox_inches="tight")
plt.show()
valid_metrics = metrics_df.dropna(subset=["RMSE"])


best_model_name = valid_metrics.iloc[0]["Model"]
best_pred = forecast_table[best_model_name]

best_model_path = OUTPUT_DIR / "figure_5.png"

plt.figure(figsize=(12, 6))
plt.plot(train.index, endog_train, label="Тренировочные")
plt.plot(test.index, endog_test, label="Тест", linewidth=2)
plt.plot(best_pred.index, best_pred.values, label=f"Forecast: {best_model_name}", linewidth=2)
plt.axvline(test.index[0], linestyle="--")
plt.title(f"Лучшая модель по  RMSE: {best_model_name}")
plt.xlabel("Дата")
plt.ylabel("Таргет")
plt.legend()
plt.tight_layout()
plt.savefig(best_model_path, dpi=300, bbox_inches="tight")
plt.show()
summary_candidates = {"arimax_summary.txt": ["arimax_res", "arima_res"],"var_summary.txt": ["var_res"],"vecm_summary.txt": ["vecm_res"],"ardl_summary.txt": ["ardl_res"],
    "uecm_summary.txt": ["uecm_res"],
"ucm_summary.txt": ["ucm_res"],
}

for filename, obj_names in summary_candidates.items():
    for obj_name in obj_names:
        if obj_name in globals():
            try:
                txt = str(globals()[obj_name].summary())
                with open(OUTPUT_DIR / filename, "w", encoding="utf-8") as f:
                    f.write(txt)
                break
            except Exception:
                pass
forecast_table.to_csv(OUTPUT_DIR / "forecast_table.csv", index=True)
error_table.to_csv(OUTPUT_DIR / "error_table.csv", index=True)
metrics_df.to_csv(OUTPUT_DIR / "metrics_table.csv", index=False)
sample_info.to_csv(OUTPUT_DIR / "sample_info.csv", index=False)
stationarity_df.to_csv(OUTPUT_DIR / "stationarity_tests.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "article_tables.xlsx") as writer:
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    forecast_table.to_excel(writer, sheet_name="forecasts", index=True)
    error_table.to_excel(writer, sheet_name="errors", index=True)
    sample_info.to_excel(writer, sheet_name="sample_info", index=False)
    stationarity_df.to_excel(writer, sheet_name="stationarity", index=False)
    desc_full.to_excel(writer, sheet_name="desc_full", index=True)

save_latex_table(metrics_df.set_index("RMSE"), OUTPUT_DIR / "metrics_table.tex",
                 caption="точность прогнозов", label="metrics")
save_latex_table(sample_info.set_index("Block"), OUTPUT_DIR / "sample_info.tex",
                 caption="sample", label="sample")
save_latex_table(stationarity_df.set_index("Series"), OUTPUT_DIR / "stationarity_тест.tex",
                 caption="стационарность", label="стационарность")


,ранжировка_RMSE,Model,N_used,MAE,RMSE,sMAPE_%,MAPE_safe_%,Directional_Accuracy_%
0,1,UCM,9,0.000061,0.000072,0.735661,0.734065,100.0
1,2,VECM,9,0.000249,0.000277,3.021210,2.986208,100.0
2,3,ARIMAX,9,0.000261,0.000289,3.046582,2.994083,100.0
3,4,VAR,9,0.000295,0.000313,3.601995,3.534118,100.0
4,5,ARDL,9,0.000481,0.000619,5.006163,5.204932,100.0
5,6,TVP-ARX,9,0.000681,0.000792,7.211758,7.549141,100.0


,Actual,ARIMAX,VAR,VECM,ARDL,UCM,TVP-ARX
date,,,,,,,
2025-05-01,0.006616,0.006703,0.006409,0.006421,0.006644,0.006537,0.006757
2025-06-01,0.007099,0.007011,0.006873,0.006904,0.007214,0.007062,0.007353
2025-07-01,0.007556,0.007366,0.007276,0.007304,0.007750,0.007497,0.007900
2025-08-01,0.008176,0.007919,0.007717,0.007792,0.008356,0.008090,0.008632
2025-09-01,0.008633,0.008219,0.008186,0.008264,0.008965,0.008544,0.009253


,Actual,ARIMAXErr,ARIMAXAE,VARErr,VARAE,VECMErr,VECMAE,ARDLErr,ARDLAE,UCMErr,UCMAE,TVP-ARXErr,TVP-ARXAE
date,,,,,,,,,,,,,
2025-05-01,0.006616,-0.000086,0.000086,0.000207,0.000207,0.000195,0.000195,-0.000027,0.000027,0.000079,0.000079,-0.000141,0.000141
2025-06-01,0.007099,0.000088,0.000088,0.000226,0.000226,0.000195,0.000195,-0.000116,0.000116,0.000037,0.000037,-0.000254,0.000254
2025-07-01,0.007556,0.000190,0.000190,0.000280,0.000280,0.000252,0.000252,-0.000194,0.000194,0.000059,0.000059,-0.000344,0.000344
2025-08-01,0.008176,0.000257,0.000257,0.000459,0.000459,0.000384,0.000384,-0.000180,0.000180,0.000086,0.000086,-0.000456,0.000456
2025-09-01,0.008633,0.000414,0.000414,0.000447,0.000447,0.000369,0.000369,-0.000332,0.000332,0.000089,0.000089,-0.000620,0.000620


,Block,Start,End,Observations
0,Full sample,2018-02-01,2026-01-01,96
1,Train,2018-04-01,2025-04-01,85
2,Test,2025-05-01,2026-01-01,9


,count,mean,std,min,25%,50%,75%,max
target_ratio,96.0,0.006947,0.002574,0.003335,0.004446,0.007082,0.009183,0.01155
rate_spread,96.0,-2.204896,5.715900,-13.470000,-7.645000,0.360000,2.542500,3.39000


,Series,ADF statistic,ADF p-value,KPSS statistic,KPSS p-value,N obs
0,target_ratio_level,-2.085259,0.250483,1.058509,0.01,96
1,target_ratio_diff1,-0.724789,0.840224,0.977236,0.01,95
2,target_ratio_diff2,-2.932385,0.041710,0.099367,0.10,94
3,rate_spread_level,-1.133626,0.701478,1.318910,0.01,96


In [ ]:


OUT = Path("thesis_outputs")
OUT.mkdir(exist_ok=True)

sample_table = pd.DataFrame({"Показатель": ["начало выборки","конец выборки","общее число наблюдений","размер обучающей выборки","размер тестовой ","горизонт  прогноза"],
    "Значение": [str(model_df.index.min().date()) if hasattr(model_df.index.min(), "date") else str(model_df.index.min()),str(model_df.index.max().date()) if hasattr(model_df.index.max(), "date") else str(model_df.index.max()),
        len(model_df),len(train),len(test),len(test)]})

display(sample_table)

sample_table.to_excel(OUT / "table_1.xlsx", index=False)

,Показатель,Значение
0,начало выборки,2018-04-01
1,конец выборки,2026-01-01
2,общее число наблюдений,94
3,размер обучающей выборки,85
4,размер тестовой,9
5,горизонт прогноза,9


In [ ]:

stationarity_table = stationarity_train_df.copy()
stationarity_table = stationarity_table.rename(columns={"series": "Ряд","level_adf_pvalue": "ADF p-value (уровень)","level_kpss_pvalue": "KPSS p-value (уровень)",
    "stationary_in_levels": "Стационарен в уровнях","chosen_diff_order": "Выбранный порядок разности"})

display(stationarity_table)

stationarity_table.to_excel(OUT / "table_2.xlsx", index=False)

,Ряд,ADF p-value (уровень),KPSS p-value (уровень),Стационарен в уровнях,Выбранный порядок разности
0,количество_предоставленных_кредитов_за_месяц_е...,0.045902,0.10,True,0
1,earlyrep_отступными,0.125752,0.01,False,1
2,overdue_debt,0.828501,0.10,False,1
3,participants_количество_действующих_кредитных_...,0.088313,0.01,False,1
4,participants_количество_кредитных_организаций_...,0.026080,0.01,False,1
5,rate_spread,0.911221,0.01,False,1
6,target_ratio,0.562387,0.01,False,1
7,total_debt,0.982723,0.01,False,1
8,безработные_15_72_тыс,0.922871,0.01,False,1
9,зп_номин_руб,0.865303,0.01,False,1


In [ ]:
forecast_table_pretty = forecast_table.copy()
error_table_pretty = error_table.copy()

display(forecast_table_pretty.head(12))
display(error_table_pretty.head(12))

with pd.ExcelWriter(OUT / "table_3.xlsx", engine="openpyxl") as writer:
    forecast_table_pretty.to_excel(writer, sheet_name="Прогнозы")
    error_table_pretty.to_excel(writer, sheet_name="Ошибки")

forecast_table_pretty.to_csv(OUT / "table_3.csv", encoding="utf-8-sig")

,Actual,ARIMAX,VAR,VECM,ARDL,UCM,TVP-ARX
date,,,,,,,
2025-05-01,0.006616,0.006703,0.006409,0.006421,0.006644,0.006537,0.006757
2025-06-01,0.007099,0.007011,0.006873,0.006904,0.007214,0.007062,0.007353
2025-07-01,0.007556,0.007366,0.007276,0.007304,0.007750,0.007497,0.007900
2025-08-01,0.008176,0.007919,0.007717,0.007792,0.008356,0.008090,0.008632
2025-09-01,0.008633,0.008219,0.008186,0.008264,0.008965,0.008544,0.009253
2025-10-01,0.008988,0.008519,0.008616,0.008729,0.009560,0.008953,0.009754
2025-11-01,0.009421,0.009105,0.009108,0.009262,0.010167,0.009425,0.010420
2025-12-01,0.009790,0.009475,0.009626,0.009800,0.010750,0.009766,0.010977
2026-01-01,0.009955,0.009740,0.010142,0.010374,0.011159,0.010091,0.011316


,Actual,ARIMAXErr,ARIMAXAE,VARErr,VARAE,VECMErr,VECMAE,ARDLErr,ARDLAE,UCMErr,UCMAE,TVP-ARXErr,TVP-ARXAE
date,,,,,,,,,,,,,
2025-05-01,0.006616,-0.000086,0.000086,0.000207,0.000207,0.000195,0.000195,-0.000027,0.000027,0.000079,0.000079,-0.000141,0.000141
2025-06-01,0.007099,0.000088,0.000088,0.000226,0.000226,0.000195,0.000195,-0.000116,0.000116,0.000037,0.000037,-0.000254,0.000254
2025-07-01,0.007556,0.000190,0.000190,0.000280,0.000280,0.000252,0.000252,-0.000194,0.000194,0.000059,0.000059,-0.000344,0.000344
2025-08-01,0.008176,0.000257,0.000257,0.000459,0.000459,0.000384,0.000384,-0.000180,0.000180,0.000086,0.000086,-0.000456,0.000456
2025-09-01,0.008633,0.000414,0.000414,0.000447,0.000447,0.000369,0.000369,-0.000332,0.000332,0.000089,0.000089,-0.000620,0.000620
2025-10-01,0.008988,0.000469,0.000469,0.000372,0.000372,0.000259,0.000259,-0.000572,0.000572,0.000035,0.000035,-0.000766,0.000766
2025-11-01,0.009421,0.000316,0.000316,0.000313,0.000313,0.000159,0.000159,-0.000746,0.000746,-0.000004,0.000004,-0.000999,0.000999
2025-12-01,0.009790,0.000315,0.000315,0.000164,0.000164,-0.000010,0.000010,-0.000960,0.000960,0.000024,0.000024,-0.001188,0.001188
2026-01-01,0.009955,0.000216,0.000216,-0.000186,0.000186,-0.000418,0.000418,-0.001204,0.001204,-0.000135,0.000135,-0.001360,0.001360


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot( train.index,train["target_ratio"],linewidth=2,label="трейн выборка ")

plt.plot(test.index,test["target_ratio"],linewidth=2,label="тестовая выборка")


plt.axvline(test.index[0],linestyle="--",linewidth=1.5,label="граница train/test")


plt.axvspan(test.index[0],test.index[-1],alpha=0.15)

plt.title("динамика таргета и разбиение на train/test")
plt.xlabel("Дата")
plt.ylabel("Таргет")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "figure_1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(forecast_table.index, forecast_table["Actual"], linewidth=2, label="Факт")

for col in forecast_table.columns:
    if col != "Actual":
        plt.plot(forecast_table.index, forecast_table[col], label=col)

plt.title("фактические и прогнозные значения на тестовом интервале")
plt.xlabel("Дата")
plt.ylabel("таргет")
plt.legend(loc="best", ncol=2)
plt.tight_layout()
plt.savefig(OUT / "figure_2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:

plt.figure(figsize=(13, 6))

for col in error_table.columns:
    plt.plot(error_table.index, error_table[col], label=col)

plt.axhline(0, linestyle="--")
plt.title("ошибки прогноза по моделям на тестовом интервале")
plt.xlabel("Дата")
plt.ylabel("ошибка прогноза")
plt.legend(loc="best", ncol=2)
plt.tight_layout()
plt.savefig(OUT / "figure_3.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top3_models = metrics_df.sort_values("RMSE")["Model"].dropna().head(3).tolist()

plt.figure(figsize=(12, 6))
plt.plot(forecast_table.index, forecast_table["Actual"], linewidth=2, label="Факт")

for m in top3_models:
    if m in forecast_table.columns:
        plt.plot(forecast_table.index, forecast_table[m], label=m)

plt.title("сравнение 3лучших моделей с фактическим рядом")
plt.xlabel("Дата")
plt.ylabel("таргет")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "figure_4.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_metrics = metrics_df.sort_values("RMSE").copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_metrics["Model"], plot_metrics["RMSE"])
plt.title("сравнение моделей по RMSE")
plt.xlabel("Модель")
plt.ylabel("RMSE")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT / "figure_5.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
ucm_resid = pd.Series(ucm_res.resid, index=endog_train.index)

plt.figure(figsize=(12, 5))
plt.plot(ucm_resid.index, ucm_resid)
plt.axhline(0, linestyle="--")
plt.title("Остатки UCM на обучающем интервале")
plt.xlabel("Дата")
plt.ylabel("Остаток")
plt.tight_layout()
plt.savefig(OUT / "figure_6.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 4))
plot_acf(ucm_resid.dropna(), lags=24)
plt.title("ACF остатков UCM")
plt.tight_layout()
plt.savefig(OUT / "figure_7.png", dpi=300, bbox_inches="tight")
plt.show()